In [1]:
import joblib
import pandas as pd
import numpy as np
from datetime import datetime

In [9]:
# Load the saved model bundle
pipeline = joblib.load("best_model.pkl")
# preprocessor = pipeline['preprocessor']

In [8]:
pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['forks', 'open_issues',
                                                   'size', 'subscribers_count',
                                                   'contributors_count',
                                                   'commits_count',
                                                   'readme_size', 'project_age',
                                                   'days_since_update',
                                                   'days_since_push',
                                                   'forks_per_day',
                                                   'issues_per_day',
                                                   'update_rate']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['language', 'license'])])),
                ('regressor',
                 MLPRegressor(hidden_layer_sizes=(128, 64), max_iter=1000,
                              random_state=42))])

In [ ]:
# def transform(df):
#     df["log_stars"] = np.log1p(df["stars"]) # log(1 + x) to avoid log(0)

#     # Parse and normalize time-related features
#     date = datetime(2025, 5, 1)
#     df["created_at"] = pd.to_datetime(df["created_at"]).dt.tz_localize(None)
#     df["updated_at"] = pd.to_datetime(df["updated_at"]).dt.tz_localize(None)
#     df["pushed_at"] = pd.to_datetime(df["pushed_at"]).dt.tz_localize(None)
#     df["project_age"] = (date - df["created_at"]).dt.days
#     df["days_since_update"] = (date - df["updated_at"]).dt.days
#     df["days_since_push"] = (date - df["pushed_at"]).dt.days

#     # Handle missing values
#     df["license"] = df["license"].fillna("None")
#     df["language"] = df["language"].fillna("Unknown")

#     # Derived rate-based features
#     df["forks_per_day"] = df["forks"] / (df["project_age"] + 1)
#     df["issues_per_day"] = df["open_issues"] / (df["project_age"] + 1)
#     df["update_rate"] = 1 / (1 + df["days_since_update"])

#     # Replace inf with NaN and drop rows with NaN
#     df.replace([np.inf, -np.inf], np.nan, inplace=True)
#     df.dropna(inplace=True)

#     # Selected features
#     features = [
#         'forks', 'watchers', 'open_issues',
#         'size', 'has_wiki', 'has_projects', 'has_downloads', 'is_fork',
#         'archived', 'language', 'license', 'subscribers_count', 
#         'contributors_count', 'commits_count', 'readme_size',
#         'project_age', 'days_since_update', 'days_since_push','update_rate'
#     ]
#     return df[features]

In [4]:
# Create a real-world sample repository
sample_repo = {
    'name': 'ml-web-app',
    'full_name': 'data-scientist/ml-web-app',
    'created_at': '2023-05-10T08:00:00Z',
    'updated_at': '2023-11-15T14:25:00Z',
    'pushed_at': '2023-11-15T14:30:00Z',
    'language': 'Python',  # Must be in encoders['language'].classes_
    'license': 'mit',      # Must be in encoders['license'].classes_
    'forks': 87,
    'watchers': 420, # same as stars, unknown
    'open_issues': 12,
    'size': 3500,
    'has_wiki': True,
    'has_projects': False,
    'has_downloads': True,
    'is_fork': False,
    'archived': False,
    'subscribers_count': 150,
    'readme_size': 1024,
    'commits_count': 85,
    'contributors_count': 12
}
sample_repo_df = pd.DataFrame([sample_repo])


In [ ]:
# Make and show prediction
X_sample = transform(sample_repo_df)

if X_sample is not None:
    y_pred = pipeline.predict(X_sample)[0]
    model = pipeline['regressor']
    print("\n=== GitHub Stars Prediction ===")
    print(f"Repository: {sample_repo['full_name']}")
    print(f"Language: {sample_repo['language']}")
    print(f"License: {sample_repo['license']}")
    print(f"Created: {sample_repo['created_at']}")
    print(f"\nPredicted Stars: {round(y_pred)}")
    
    # Show confidence (for regression models)
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_sample)[0]
        print(f"Confidence: {max(proba)*100:.1f}%")
    
    # Show available categories
    print("\nModel was trained with:")
    # print(f"Languages: {list(encoders['language'].classes_)}")
    # print(f"Licenses: {list(encoders['license'].classes_)}")
    print(f"Features ({len(X_sample.columns)}): {X_sample.columns}")
else:
    print("Prediction failed due to preprocessing error")


=== GitHub Stars Prediction ===
Repository: data-scientist/ml-web-app
Language: Python
License: mit
Created: 2023-05-10T08:00:00Z

Predicted Stars: 3096812

Model was trained with:
Features (21): Index(['forks', 'watchers', 'open_issues', 'size', 'has_wiki', 'has_projects',
       'has_downloads', 'is_fork', 'archived', 'language', 'license',
       'subscribers_count', 'contributors_count', 'commits_count',
       'readme_size', 'project_age', 'days_since_update', 'days_since_push',
       'forks_per_day', 'issues_per_day', 'update_rate'],
      dtype='object')


# Predict 5 real-world GitHub repositories and rank them

## Get random repositories

In [ ]:
import requests
import pandas as pd
import time
import random
import json

# GitHub Token
GITHUB_TOKEN = 'github_pat_3434fc434------' # Paste your token
HEADERS = {'Authorization': f'token {GITHUB_TOKEN}'}
MIN_STARS = 5000

# Load training data
train_df = pd.read_csv('github_repo_features_new.csv')
train_repo_names = set(train_df['full_name'].tolist())

def get_random_repo():
    """search for a random repo on GitHub"""
    random_keyword = random.choice(['python', 'data', 'ai', 'react', 'cloud', 'web', 'api', 'linux'])
    url = f'https://api.github.com/search/repositories?q={random_keyword}+stars:>={MIN_STARS}&sort=updated&order=desc&per_page=30'
    response = requests.get(url, headers=HEADERS)
    
    if response.status_code != 200:
        print(f"Error: {response.status_code}")
        return None
    
    candidates = response.json().get('items', [])
    for repo in candidates:
        if repo['full_name'] not in train_repo_names:
            return repo
    return None

def get_test_repos(n=5):
    """Get n test repositories from GitHub"""
    test_repos = []
    attempts = 0
    while len(test_repos) < n and attempts < 100:
        repo = get_random_repo()
        if repo and repo['full_name'] not in [r['full_name'] for r in test_repos]:
            test_repos.append(repo)
            print(f"Found test repo: {repo['full_name']}")
        attempts += 1
        time.sleep(1)
    return test_repos

# Get test repositories
test_repos = get_test_repos(n=5)
with open('github_test_repos_raw.json', 'w', encoding='utf-8') as f:
    json.dump(test_repos, f, ensure_ascii=False, indent=4)

Found test repo: meshery/meshery
Found test repo: teambit/bit
Found test repo: PrismarineJS/mineflayer
Found test repo: jesse-ai/jesse
Found test repo: CVHub520/X-AnyLabeling


In [2]:
import json
# load the row data
with open('github_test_repos_raw.json', 'r', encoding='utf-8') as f:
    test_repos_loaded = json.load(f)

print(type(test_repos_loaded))  # <class 'list'>
print(type(test_repos_loaded[0]))  # <class 'dict'>

<class 'list'>
<class 'dict'>


In [ ]:
import re
import requests
import pandas as pd

GITHUB_TOKEN = 'github_pat_3434fc434------' # Paste your token
HEADERS = {'Authorization': f'token {GITHUB_TOKEN}'}
def get_count_with_pagination(url):
    try:
        first_page_url = f"{url}?per_page=1&page=1"
        response = requests.get(first_page_url, headers=HEADERS)

        if response.status_code != 200:
            return 0

        # Parse Link header to get the last page number
        link_header = response.headers.get('Link', '')
        match = re.search(r'page=(\d+)>; rel="last"', link_header)

        if match:
            return int(match.group(1))
        else:
            # If no pagination, return the count of items in the response
            return len(response.json())  # 0 或 1
    except:
        return 0

def get_contributor_count(url):
    url = f"{url}?anon=true&per_page=1&page=1002"  # Use a very large page number to trick GitHub into returning to the Link header

    try:
        response = requests.get(url, headers=HEADERS)
        if response.status_code != 200:
            return 0

        # Parse Link header to get the last page number
        link_header = response.headers.get('Link', '')
        match = re.search(r'page=(\d+)>; rel="last"', link_header)
        if match:
            last_page = int(match.group(1))
            return last_page 
        else:
            # If no pagination, return the count of items in the response
            data = response.json()
            return len(data) if isinstance(data, list) else 0

    except Exception as e:
        return 0

def get_readme_size(repo_full_name):
    try:
        url = f'https://api.github.com/repos/{repo_full_name}/readme'
        response = requests.get(url, headers=HEADERS)
        if response.status_code == 200:
            return response.json().get('size', 0)
    except:
        pass
    return 0

def extract_features(repo_data):
    features = {
        'name': repo_data['name'],
        'full_name': repo_data['full_name'],
        'stars': repo_data['stargazers_count'],
        'forks': repo_data['forks_count'],
        'watchers': repo_data['watchers_count'],
        'open_issues': repo_data['open_issues_count'],
        'size': repo_data['size'],
        'has_wiki': int(repo_data['has_wiki']),
        'has_projects': int(repo_data['has_projects']),
        'has_downloads': int(repo_data['has_downloads']),
        'is_fork': int(repo_data['fork']),
        'archived': int(repo_data['archived']),
        'language': repo_data['language'],
        'license': repo_data['license']['key'] if repo_data['license'] else None,
        'created_at': repo_data['created_at'],
        'updated_at': repo_data['updated_at'],
        'pushed_at': repo_data['pushed_at'],

        # New features
        'has_description': int(bool(repo_data['description'])),
        'has_homepage': int(bool(repo_data['homepage'])),
        'topic_count': len(repo_data['topics']) if 'topics' in repo_data and isinstance(repo_data['topics'], list) else 0,
        'has_discussions': int(repo_data.get('has_discussions', False)),
        'is_template': int(repo_data.get('is_template', False)),
        'allow_forking': int(repo_data.get('allow_forking', True)),
        'visibility': repo_data.get('visibility', 'public'),

        # Counts with pagination
        'subscribers_count': get_count_with_pagination(repo_data['subscribers_url']),
        'contributors_count': get_contributor_count(repo_data['contributors_url']),
        'commits_count': get_count_with_pagination(repo_data['commits_url'].split('{')[0]),
        'readme_size': get_readme_size(repo_data['full_name'])
    }
    return features

# Process all repositories
c=1
features_list = []
for repo in test_repos_loaded:
    features = extract_features(repo)
    features_list.append(features)
    print(c)
    c+=1
    print(features)

features_df = pd.DataFrame(features_list)
features_df.to_csv('github_test_repos.csv', index=False)

1
{'name': 'meshery', 'full_name': 'meshery/meshery', 'stars': 7075, 'forks': 2267, 'watchers': 7075, 'open_issues': 652, 'size': 1059785, 'has_wiki': 0, 'has_projects': 1, 'has_downloads': 1, 'is_fork': 0, 'archived': 0, 'language': 'JavaScript', 'license': 'apache-2.0', 'created_at': '2018-11-14T13:41:00Z', 'updated_at': '2025-05-18T14:21:37Z', 'pushed_at': '2025-05-18T14:21:21Z', 'has_description': 1, 'has_homepage': 1, 'topic_count': 20, 'has_discussions': 1, 'is_template': 0, 'allow_forking': 1, 'visibility': 'public', 'subscribers_count': 52, 'contributors_count': 1284, 'commits_count': 42942, 'readme_size': 30790}
2
{'name': 'bit', 'full_name': 'teambit/bit', 'stars': 18065, 'forks': 937, 'watchers': 18065, 'open_issues': 44, 'size': 195067, 'has_wiki': 0, 'has_projects': 1, 'has_downloads': 1, 'is_fork': 0, 'archived': 0, 'language': 'TypeScript', 'license': 'other', 'created_at': '2017-01-22T14:51:43Z', 'updated_at': '2025-05-18T14:30:22Z', 'pushed_at': '2025-05-18T14:30:18Z',

## Rank 5 repos with best model according to their star count

In [4]:
import joblib
import pandas as pd
import numpy as np
from datetime import datetime

# Load the saved model bundle
pipeline = joblib.load("best_model_cv.pkl")
# preprocessor = pipeline['preprocessor']

In [5]:
def transform(df):
    date = datetime(2025, 5, 17)
    
    # Parse and normalize time-related features
    df["project_age"] = (date.date() - df["created_at"].dt.date).apply(lambda x: x.days)
    df["days_since_update"] = (date.date() - df["updated_at"].dt.date).apply(lambda x: x.days)
    df["days_since_push"] = (date.date() - df["pushed_at"].dt.date).apply(lambda x: x.days)

    # Handle missing values
    df["license"] = df["license"].fillna("None")
    df["language"] = df["language"].fillna("Unknown")

    # Derived rate-based features
    df["forks_per_day"] = df["forks"] / (df["project_age"] + 1)
    df["issues_per_day"] = df["open_issues"] / (df["project_age"] + 1)
    df["update_rate"] = 1 / (1 + df["days_since_update"])

    features = [
            'forks', 'open_issues',
       'size', 'has_wiki', 'has_projects', 'has_downloads', 
       'archived', 'language', 'license', 
       'has_description', 'has_homepage', 'topic_count',
       'has_discussions', 'is_template', 
       'subscribers_count', 'contributors_count', 'commits_count',
       'readme_size', 'project_age', 'days_since_push',
       'forks_per_day', 'issues_per_day'
        ]
    
    return df[features]

In [7]:
# Load the test data
test_df = pd.read_csv('github_test_repos.csv', parse_dates=["created_at", "updated_at", "pushed_at"])

# Transform the test data
X_test = transform(test_df)

if X_test is not None:
    # Make predictions
    y_preds = pipeline.predict(X_test)

    # Add predictions to the DataFrame
    test_df['predicted_stars'] = y_preds

    # Sort the DataFrame by predicted stars
    test_df_sorted = test_df.sort_values(by='predicted_stars', ascending=False)

    # Display the results
    print("\n=== GitHub Stars Prediction (Sorted) ===")
    for i, row in test_df_sorted.iterrows():
        print(f"\n--- Repository {i+1} ---")
        print(f"Name: {row['full_name']}")
        print(f"Language: {row['language']}")
        print(f"License: {row['license']}")
        print(f"Created At: {row['created_at']}")
        print(f"Predicted Stars: {round(row['predicted_stars'])}")
else:
    print("Prediction failed due to preprocessing error.")


=== GitHub Stars Prediction (Sorted) ===

--- Repository 1 ---
Name: meshery/meshery
Language: JavaScript
License: apache-2.0
Created At: 2018-11-14 13:41:00+00:00
Predicted Stars: 33079

--- Repository 3 ---
Name: PrismarineJS/mineflayer
Language: JavaScript
License: mit
Created At: 2011-01-23 09:41:55+00:00
Predicted Stars: 32663

--- Repository 4 ---
Name: jesse-ai/jesse
Language: JavaScript
License: mit
Created At: 2018-11-09 10:38:44+00:00
Predicted Stars: 32541

--- Repository 2 ---
Name: teambit/bit
Language: TypeScript
License: other
Created At: 2017-01-22 14:51:43+00:00
Predicted Stars: 32447

--- Repository 5 ---
Name: CVHub520/X-AnyLabeling
Language: Python
License: gpl-3.0
Created At: 2023-05-23 08:14:30+00:00
Predicted Stars: 28118
